# Policy Gradient Methods in Reinforcement Learning

This notebook provides an introduction to policy gradient methods, a fundamental class of algorithms in reinforcement learning that directly optimize the policy by computing gradients of the expected return with respect to the policy parameters.

## Table of Contents

1. [Introduction](#introduction)
2. [Policy Gradient Theorem](#policy-gradient-theorem)
3. [REINFORCE Algorithm](#reinforce-algorithm)
4. [Advantage Functions](#advantage-functions)
5. [Actor-Critic Methods](#actor-critic-methods)
6. [Practical Implementation](#practical-implementation)

## Introduction

### Why Policy Gradient Methods?

Policy gradient methods represent a major paradigm in reinforcement learning. Unlike value-based methods (e.g., Q-learning) that learn the value of taking an action in a state, policy gradient methods directly learn a parametrized policy $\pi_\theta(a|s)$ that maps states to action probabilities.

**Key advantages:**
- Can handle continuous action spaces naturally
- Guaranteed convergence to a local optimum (under certain conditions)
- Better theoretical convergence properties than value-based methods
- Can learn stochastic policies, which are useful for exploration and some problem structures

### Policy Parametrization

We parametrize the policy as $\pi_\theta(a|s) = P(a|s; \theta)$, where:
- $s$ is the current state
- $a$ is the action to take
- $\theta$ is the parameter vector (e.g., neural network weights)

For discrete action spaces, we often use softmax:
$$\pi_\theta(a|s) = \frac{e^{h_\theta(s,a)}}{\sum_{a'} e^{h_\theta(s,a')}}$$

where $h_\theta(s,a)$ is some function of the state and action (e.g., output of a neural network).

## Policy Gradient Theorem

### The Core Insight

The policy gradient theorem provides the foundation for all policy gradient methods. It tells us how to compute the gradient of the expected return with respect to the policy parameters.

**Theorem (Policy Gradient Theorem):**
$$\nabla_\theta J(\theta) = \mathbb{E}_{s \sim \rho(s)} \left[ \sum_a \nabla_\theta \pi_\theta(a|s) Q^\pi(s,a) \right]$$

where:
- $J(\theta) = \mathbb{E}[R_0]$ is the expected return (objective function)
- $\rho(s)$ is the stationary distribution of states under policy $\pi_\theta$
- $Q^\pi(s,a)$ is the action-value function (expected return starting from state $s$, taking action $a$, then following policy $\pi$)
- $\nabla_\theta \pi_\theta(a|s)$ is the gradient of the policy

### Rearranging with the Log-Derivative Trick

We can rewrite using the identity: $\nabla_\theta \pi_\theta(a|s) = \pi_\theta(a|s) \nabla_\theta \log \pi_\theta(a|s)$

$$\nabla_\theta J(\theta) = \mathbb{E}_{s,a} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot Q^\pi(s,a) \right]$$

This is the **fundamental equation** for policy gradient methods. The term $\nabla_\theta \log \pi_\theta(a|s)$ is called the **score function** or **policy gradient**.

## REINFORCE Algorithm

### The Algorithm

REINFORCE (REward Increment = Non-negative Factor times Offset Reinforcement) is the simplest policy gradient algorithm. It directly applies the policy gradient theorem by estimating $Q^\pi(s,a)$ using the actual return $G_t$ from a single trajectory.

**Algorithm:**

1. Initialize policy parameters $\theta$ randomly
2. Repeat:
   - Generate a trajectory using policy $\pi_\theta$: $s_0, a_0, r_1, s_1, a_1, r_2, ..., s_T, a_T, r_{T+1}$
   - For each timestep $t = 0, 1, ..., T$:
     - Compute return: $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_{k+1}$ (or $G_t = R_t$ for undiscounted)
     - Perform gradient ascent: $\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(a_t|s_t) G_t$

### Why This Works

The key insight is that:
- $\nabla_\theta \log \pi_\theta(a_t|s_t)$ points in the direction that increases the log probability of action $a_t$
- Multiplying by $G_t$ (the return) makes the update proportional to how good the outcome was
- If $G_t > 0$, we increase the probability of $a_t$ (reinforcement)
- If $G_t < 0$, we decrease the probability of $a_t$ (punishment)

### Advantages and Disadvantages

**Advantages:**
- Simple to understand and implement
- Guaranteed convergence to a local optimum (under certain conditions)
- Works with any differentiable policy

**Disadvantages:**
- High variance: relies on the sampled return from a single trajectory
- Slow convergence due to high variance
- Sample inefficient: needs many trajectories to get good estimates

## Advantage Functions

### The Problem with REINFORCE

The fundamental issue with REINFORCE is **high variance**. Using the return $G_t$ as an estimate of $Q^\pi(s_t, a_t)$ means:
- Positive returns reinforce the action (even if it was just lucky)
- Negative returns punish the action (even if following a generally good trajectory)

### Advantage Functions

We can reduce variance by using an **advantage function**:
$$A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$$

where $V^\pi(s) = \mathbb{E}_a[Q^\pi(s,a)]$ is the state value function.

**Intuition:** The advantage tells us how much better taking action $a$ is compared to the average action in state $s$. This naturally centers the returns and reduces variance.

### Variance Reduction

Why is this better?
- Baseline subtraction: by subtracting $V^\pi(s_t)$, we remove the component of return that doesn't depend on the action
- This doesn't change the expected value (the gradient is unbiased)
- But it significantly reduces the variance of the gradient estimator

**Gradient with advantage:**
$$\nabla_\theta J(\theta) = \mathbb{E}_{s,a} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot A^\pi(s,a) \right]$$

### Temporal Difference (TD) Advantage

In practice, we estimate the advantage using TD residuals:
$$\hat{A}(s_t, a_t) = r_t + \gamma V^\pi(s_{t+1}) - V^\pi(s_t)$$

This requires learning a value function $V^\pi(s)$ in addition to the policy.

## Actor-Critic Methods

### Combining Policy and Value Learning

Actor-Critic methods combine:
1. **Actor:** The policy $\pi_\theta(a|s)$ that chooses actions
2. **Critic:** A value function $V_\phi(s)$ that estimates state values

The critic helps reduce variance by providing better advantage estimates.

### Actor-Critic Update Rule

At each timestep $t$:

1. **Actor (Policy) Update:**
   $$\theta \leftarrow \theta + \alpha_\theta \nabla_\theta \log \pi_\theta(a_t|s_t) \hat{A}_t$$

2. **Critic (Value) Update:**
   $$\phi \leftarrow \phi + \alpha_\phi \nabla_\phi \mathcal{L}(\phi)$$
   
   where $\mathcal{L}(\phi) = (V_\phi(s_t) - (r_t + \gamma V_\phi(s_{t+1})))^2$ is the TD error squared

3. **Advantage Estimate:**
   $$\hat{A}_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$$

### Advantages

- **Lower variance:** Critic provides a better baseline than just using 0
- **Online learning:** Can update after each step (or mini-batch), not just at episode end
- **Better sample efficiency:** Value function learning provides extra signal
- **Faster convergence:** Due to lower variance and better baselines

### Examples of Actor-Critic Algorithms

- **A3C (Asynchronous Advantage Actor-Critic):** Uses multiple parallel environments
- **A2C:** Synchronous version of A3C
- **DDPG (Deep Deterministic Policy Gradient):** For continuous control
- **TD3:** Improved version of DDPG
- **SAC (Soft Actor-Critic):** Entropy-regularized actor-critic for sample efficiency

## Practical Implementation

### Example: Simple REINFORCE on CartPole

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gym

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

class PolicyNetwork(nn.Module):
    """A simple neural network policy."""
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, action_dim)
        
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        action_logits = self.fc2(x)
        return action_logits

class REINFORCEAgent:
    """REINFORCE algorithm implementation."""
    def __init__(self, state_dim, action_dim, learning_rate=1e-2):
        self.policy = PolicyNetwork(state_dim, action_dim)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)
        self.action_dim = action_dim
        
    def select_action(self, state):
        """Select an action using the current policy."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action_logits = self.policy(state_tensor)
        action_probs = torch.softmax(action_logits, dim=-1)
        dist = Categorical(action_probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob
    
    def update(self, trajectory):
        """
        Update policy using a full trajectory.
        
        trajectory: list of (state, action, log_prob, reward) tuples
        """
        # Compute returns (cumulative discounted rewards)
        returns = []
        G = 0
        gamma = 0.99
        
        # Compute returns in reverse order
        for state, action, log_prob, reward in reversed(trajectory):
            G = reward + gamma * G
            returns.insert(0, G)
        
        returns = torch.FloatTensor(returns)
        # Normalize returns for lower variance
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Compute policy gradient
        policy_loss = 0
        log_probs = torch.stack([log_prob for _, _, log_prob, _ in trajectory])
        
        # L = -sum(log_prob * return)
        policy_loss = -(log_probs * returns).sum()
        
        # Gradient ascent
        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()
        
        return policy_loss.item()

# Example: Train on CartPole-v1
env = gym.make('CartPole-v1', new_step_api=False)
agent = REINFORCEAgent(state_dim=4, action_dim=2, learning_rate=1e-2)

num_episodes = 100
episode_returns = []

for episode in range(num_episodes):
    state, _ = env.reset()
    trajectory = []
    episode_return = 0
    
    # Collect trajectory
    for t in range(500):  # Max 500 steps per episode
        action, log_prob = agent.select_action(state)
        next_state, reward, done, info, _ = env.step(action)
        trajectory.append((state, action, log_prob, reward))
        episode_return += reward
        
        if done:
            break
        
        state = next_state
    
    # Update policy
    loss = agent.update(trajectory)
    episode_returns.append(episode_return)
    
    if (episode + 1) % 20 == 0:
        avg_return = np.mean(episode_returns[-20:])
        print(f"Episode {episode+1}: Avg Return = {avg_return:.2f}")

env.close()
print(f"\nFinal average return (last 10 episodes): {np.mean(episode_returns[-10:]):.2f}")

### Example: Actor-Critic Method

In [ ]:
class ValueNetwork(nn.Module):
    """A simple neural network for state value estimation."""
    def __init__(self, state_dim, hidden_dim=128):
        super(ValueNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)  # Output single value
        
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        value = self.fc2(x)
        return value

class ActorCriticAgent:
    """Actor-Critic algorithm implementation."""
    def __init__(self, state_dim, action_dim, learning_rate=1e-2):
        self.actor = PolicyNetwork(state_dim, action_dim)
        self.critic = ValueNetwork(state_dim)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=learning_rate)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=learning_rate)
        self.gamma = 0.99
        
    def select_action(self, state):
        """Select an action using the current policy."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action_logits = self.actor(state_tensor)
        action_probs = torch.softmax(action_logits, dim=-1)
        dist = Categorical(action_probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob, state_tensor
    
    def update(self, state, action, log_prob, reward, next_state, done, prev_state_tensor):
        """Update actor and critic using single transition."""
        state_tensor = prev_state_tensor
        next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0)
        
        # Compute TD target and advantage
        with torch.no_grad():
            value_s = self.critic(state_tensor)
            value_next = self.critic(next_state_tensor)
            if done:
                td_target = torch.tensor([[reward]], dtype=torch.float32)
            else:
                td_target = torch.tensor([[reward + self.gamma * value_next.item()]], dtype=torch.float32)
            advantage = td_target - value_s
        
        # Actor update
        actor_loss = -(log_prob * advantage.detach())
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # Critic update
        critic_loss = advantage.pow(2).mean()
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        return actor_loss.item(), critic_loss.item()

# Train Actor-Critic on CartPole
env = gym.make('CartPole-v1', new_step_api=False)
agent = ActorCriticAgent(state_dim=4, action_dim=2, learning_rate=1e-2)

num_episodes = 100
episode_returns = []

for episode in range(num_episodes):
    state, _ = env.reset()
    episode_return = 0
    
    for t in range(500):
        action, log_prob, state_tensor = agent.select_action(state)
        next_state, reward, done, info, _ = env.step(action)
        
        agent.update(state, action, log_prob, reward, next_state, done, state_tensor)
        episode_return += reward
        
        if done:
            break
        
        state = next_state
    
    episode_returns.append(episode_return)
    
    if (episode + 1) % 20 == 0:
        avg_return = np.mean(episode_returns[-20:])
        print(f"Episode {episode+1}: Avg Return = {avg_return:.2f}")

env.close()
print(f"\nFinal average return (last 10 episodes): {np.mean(episode_returns[-10:]):.2f}")

## Key Takeaways

1. **Policy Gradient Theorem:** Provides the mathematical foundation for policy gradient methods by showing how to compute gradients of the expected return.

2. **REINFORCE:** The simplest policy gradient algorithm, but suffers from high variance.

3. **Advantage Functions:** Reduce variance by using baseline subtraction. Advantage = Q-value minus baseline.

4. **Actor-Critic Methods:** Combine policy learning (actor) with value function learning (critic) for lower variance and better sample efficiency.

5. **Practical Considerations:**
   - Normalize returns/advantages to reduce variance
   - Use neural networks to parametrize both policy and value function
   - Can train online (after each step) or batch-wise (after episode)
   - Need careful tuning of learning rates and network architectures

## Further Reading

- Sutton & Barto (2018): "Reinforcement Learning: An Introduction" - Chapters 13-14
- Schulman et al. (2015): "Policy Gradient Methods for Reinforcement Learning"
- Mnih et al. (2016): "Asynchronous Methods for Deep Reinforcement Learning" (A3C)
- Haarnoja et al. (2018): "Soft Actor-Critic" (SAC)